In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score, precision_recall_fscore_support, classification_report

FEATURES_DIR = Path('../results/features')
FIG_DIR = Path('../results/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(FEATURES_DIR / 'features_train.csv')

X = df.select_dtypes('number').replace([np.inf, -np.inf], np.nan).fillna(0)
if 'forma_aspect_ratio' in X.columns:
    X['forma_aspect_ratio'] = X['forma_aspect_ratio'].clip(upper=5)
y = df['clase'].values
clases = sorted(np.unique(y))

Xs = StandardScaler().fit_transform(X)
print(f'{Xs.shape[0]} muestras, {Xs.shape[1]} features, {len(clases)} clases')

In [ ]:
colores = plt.cm.tab10(np.linspace(0, 1, len(clases)))
cmap = {c: colores[i] for i, c in enumerate(clases)}

proj_pca = PCA(n_components=2, random_state=0).fit_transform(Xs)
var_exp = PCA(n_components=2, random_state=0).fit(Xs).explained_variance_ratio_
proj_tsne = TSNE(n_components=2, perplexity=30, random_state=0, init='pca').fit_transform(Xs)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for c in clases:
    m = y == c
    axes[0].scatter(proj_pca[m, 0], proj_pca[m, 1], c=[cmap[c]], label=c, s=15, alpha=0.5)
    axes[1].scatter(proj_tsne[m, 0], proj_tsne[m, 1], c=[cmap[c]], label=c, s=15, alpha=0.5)
axes[0].set_title(f'PCA (var explicada: {var_exp.sum():.1%})')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
axes[1].set_title('t-SNE')
axes[1].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
plt.suptitle('Separabilidad visual de las 9 clases (train)', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / '04_separabilidad_pca_tsne.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
modelos = {
    'RandomForest': RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1),
    'SVM-RBF': SVC(kernel='rbf', C=10, gamma=0.01, random_state=0),
}

preds = {}
for nombre, modelo in modelos.items():
    preds[nombre] = cross_val_predict(modelo, Xs, y, cv=5, n_jobs=-1)
    print(f'=== {nombre} ===')
    print(classification_report(y, preds[nombre], digits=3))


def tabla_detalle_clase(yp, nombre, archivo):
    rep = classification_report(y, yp, output_dict=True)
    filas = {}
    for c in clases:
        filas[c] = {
            'Precision': rep[c]['precision'],
            'Recall': rep[c]['recall'],
            'F1': rep[c]['f1-score'],
            'Soporte': int(rep[c]['support']),
        }
    df_det = pd.DataFrame(filas).T
    df_det.index.name = 'Clase'
    df_det.loc['MACRO AVG'] = [
        rep['macro avg']['precision'], rep['macro avg']['recall'],
        rep['macro avg']['f1-score'], int(rep['macro avg']['support']),
    ]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.axis('off')
    datos = df_det.copy()
    for col in datos.columns:
        datos[col] = [f'{int(v)}' if col == 'Soporte' else f'{v:.3f}' for v in datos[col]]

    tabla = ax.table(cellText=datos.values, rowLabels=datos.index,
                     colLabels=datos.columns, cellLoc='center', rowLoc='center', loc='center')
    tabla.auto_set_font_size(False)
    tabla.set_fontsize(11)
    tabla.scale(1.2, 1.7)

    # encabezados de columna
    for j in range(len(datos.columns)):
        tabla[0, j].set_facecolor('#2c3e50')
        tabla[0, j].set_text_props(color='white', fontweight='bold')
    # encabezados de fila (clases)
    for i in range(len(datos.index)):
        tabla[i + 1, -1].set_facecolor('#34495e')
        tabla[i + 1, -1].set_text_props(color='white', fontweight='bold')

    # fila de promedio resaltada
    n = len(datos.index)
    for j in range(len(datos.columns)):
        tabla[n, j].set_facecolor('#f9e79f')
        tabla[n, j].set_text_props(fontweight='bold')

    # columna F1 en semaforo
    col_f1 = list(datos.columns).index('F1')
    for i, c in enumerate(clases):
        val = df_det.loc[c, 'F1']
        color = '#a9dfbf' if val >= 0.8 else '#f9e79f' if val >= 0.6 else '#f5b7b1'
        tabla[i + 1, col_f1].set_facecolor(color)

    ax.set_title(f'Métricas por clase — {nombre} (CV 5-fold)',
                 fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig(FIG_DIR / archivo, dpi=200, bbox_inches='tight')
    plt.show()


# genera una tabla por modelo
for nombre, yp in preds.items():
    archivo = f'04_detalle_clase_{nombre.lower().replace("-", "")}.png'
    tabla_detalle_clase(yp, nombre, archivo)

In [ ]:
for nombre, cmap_color in [('RandomForest', 'Blues'), ('SVM-RBF', 'Greens')]:
    fig, ax = plt.subplots(figsize=(9, 8))
    cm = confusion_matrix(y, preds[nombre], labels=clases)
    ConfusionMatrixDisplay(cm, display_labels=clases).plot(ax=ax, cmap=cmap_color, xticks_rotation=45, colorbar=False)
    ax.set_title(f'Matriz de confusión ({nombre}, CV 5-fold)')
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'04_matriz_confusion_{nombre.lower().replace("-", "")}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
clf = RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1).fit(Xs, y)
imp = pd.Series(clf.feature_importances_, index=X.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
imp.head(20).iloc[::-1].plot.barh(ax=ax, color='steelblue')
ax.set_title('Top 20 features por importancia (RandomForest)')
plt.tight_layout()
plt.savefig(FIG_DIR / '04_importancia_features.png', dpi=150, bbox_inches='tight')
plt.show()

cm_svm = confusion_matrix(y, preds['SVM-RBF'], labels=clases)
cm_norm = cm_svm / cm_svm.sum(axis=1, keepdims=True)
np.fill_diagonal(cm_norm, 0)
print('Confusiones mas fuertes SVM (real -> predicha):')
for _ in range(6):
    i, j = np.unravel_index(cm_norm.argmax(), cm_norm.shape)
    print(f'  {clases[i]:12s} -> {clases[j]:12s}: {cm_norm[i, j]:.1%}')
    cm_norm[i, j] = 0